In [20]:
import pandas as pd
import polars as pl
from datetime import date
from rake_nltk import Rake
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Loading to db
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 
import psycopg2
from sqlalchemy import create_engine 

In [22]:
rp = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/data/raw/reddit_pubmed_data.csv")

In [ ]:
rp['journal'].unique()
# 17682

<ArrowStringArray>
[                                                         'Annals of surgery',
                                                   'Science (New York, N.Y.)',
                                                  'The Journal of physiology',
                                       'California state journal of medicine',
                                          'Journal of anatomy and physiology',
                  'American journal of public health (New York, N.Y. : 1912)',
 'Beitrage zur Klinik der Tuberkulose und spezifischen Tuberkulose-Forschung',
                      'Transactions of the American Ophthalmological Society',
      'The American journal of obstetrics and diseases of women and children',
                                       'The British journal of ophthalmology',
 ...
                            'Polymer science & technology (Washington, D.C.)',
                                                          'NPJ metamaterials',
                            

In [ ]:
rp.info()
# 2,797,950 rows

<class 'pandas.DataFrame'>
RangeIndex: 2797950 entries, 0 to 2797949
Data columns (total 8 columns):
 #   Column    Dtype  
---  ------    -----  
 0   title     str    
 1   abstract  str    
 2   journal   str    
 3   date      str    
 4   authors   str    
 5   doi       str    
 6   keyword   str    
 7   year      float64
dtypes: float64(1), str(7)
memory usage: 4.2 GB


In [ ]:
rp['year'].min()

np.float64(1786.0)

In [ ]:
rp.isna().sum()

title            0
abstract    472647
journal       1881
date            13
authors          0
doi         557210
keyword          0
year            13
dtype: int64

In [ ]:
rp[['title']].value_counts().head()

title                                  
[Not Available].                           840
Drugs and Lactation Database (LactMed®)    107
Polycystic ovary syndrome.                  99
Reply.                                      96
Emergency contraception.                    82
Name: count, dtype: int64

## The two different datasets

### 1. Duplicate rows: Every paper has multiple keywords (this is the original dataset)

In [ ]:
rp_k = rp
rp_k = rp_k.drop(columns=['abstract', 'journal', 'date', 'authors'])

In [ ]:
# To warehouse
# Connect to the database
'''
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

rp_k.to_sql("pubmed_key", engine, if_exists="append", index=False, method="multi", chunksize=100)
# 2797950
'''

'\nconn = psycopg2.connect(\n    dbname=os.getenv("DBNAME"),\n    user=os.getenv("DBUSER"),\n    password=os.getenv("DBPASSWORD"),\n    port=os.getenv("DBPORT"),\n    host=os.getenv("DBHOST")\n)\nconn_string=os.getenv("CONNSTRING")\n\n# Create engine\nengine = create_engine(\n    f"postgresql+psycopg2://{os.getenv(\'DBUSER\')}:{os.getenv(\'DBPASSWORD\')}"\n    f"@{os.getenv(\'DBHOST\')}:{os.getenv(\'DBPORT\')}/{os.getenv(\'DBNAME\')}"\n)\n\nrp_k.to_sql("pubmed_key", engine, if_exists="append", index=False, method="multi", chunksize=100)\n# 2797950\n'

### 2. Unique rows for each paper: Keywords are in a list

In [23]:
rp.duplicated(subset=['title', 'doi']).sum()

np.int64(562764)

In [24]:
rp[rp['doi'] == '10.1111/j.1600-0412.2012.01385.x'].head(5)

,title,abstract,journal,date,authors,doi,keyword,year
1413486,Pregnancy outcomes and the effect of metformin...,This article is a review of the literature ass...,Acta obstetricia et gynecologica Scandinavica,2012-03-02,"['Ghina SGhazeeri', 'Anwar HNassar', 'ZeinaYou...",10.1111/j.1600-0412.2012.01385.x,birth control and mood,2012.0
1414050,Pregnancy outcomes and the effect of metformin...,This article is a review of the literature ass...,Acta obstetricia et gynecologica Scandinavica,2012-03-02,"['Ghina SGhazeeri', 'Anwar HNassar', 'ZeinaYou...",10.1111/j.1600-0412.2012.01385.x,ovarian cyst,2012.0
1414935,Pregnancy outcomes and the effect of metformin...,This article is a review of the literature ass...,Acta obstetricia et gynecologica Scandinavica,2012-03-02,"['Ghina SGhazeeri', 'Anwar HNassar', 'ZeinaYou...",10.1111/j.1600-0412.2012.01385.x,PCOS,2012.0
1415008,Pregnancy outcomes and the effect of metformin...,This article is a review of the literature ass...,Acta obstetricia et gynecologica Scandinavica,2012-03-02,"['Ghina SGhazeeri', 'Anwar HNassar', 'ZeinaYou...",10.1111/j.1600-0412.2012.01385.x,polycystic ovary syndrome,2012.0
1417048,Pregnancy outcomes and the effect of metformin...,This article is a review of the literature ass...,Acta obstetricia et gynecologica Scandinavica,2012-03-02,"['Ghina SGhazeeri', 'Anwar HNassar', 'ZeinaYou...",10.1111/j.1600-0412.2012.01385.x,medical abortion,2012.0


In [25]:
# Create dataset that has the list of keywords for one publication
rp_g = rp.groupby(['title', 'doi'], as_index = False, sort = False, dropna = False).agg({'abstract':'first', 'journal':'first', 'date':'first', 'authors':'first', 'year':'first', 'keyword': lambda x: list(x.dropna().unique())})

In [26]:
rp_g.duplicated(subset=['title']).sum()

np.int64(19714)

### To CSV

In [ ]:
rp_g.to_csv('key_pubmed.csv', index=False)